# Module 1 — Web Data & LLM (RAG-Based Q&A)

Welcome! In this notebook you will build a **Retrieval-Augmented Generation (RAG)** pipeline on top of European Open Web Search data.

**What does that mean in plain English?**  

When you ask an AI model a question, we first *search* a collection of real web pages, pull the most relevant passages, and then let the AI formulate an answer grounded in those passages — with source URLs you can click and verify.

```
Your Question
     │
     ▼
  Search (retrieve relevant web pages)
     │
     ▼
  LLM reads pages + writes answer
     │
     ▼
  Grounded Answer with Sources
```

---

**Sections at a glance**

| # | Title | What you will do |
|---|-------|------------------|
| 0 | Setup & Sanity Check | Install packages, load secrets, verify LLM connectivity |
| 1 | Data Acquisition | Download Open Web Index (OWI) data with `owilix` |
| 2 | Data Processing | Clean and normalise raw web records |
| 3 | Indexing with MOSAIC | Build a local search index |
| 4 | Retrieval | Query local index, remote index, and ourrs.eu |
| 5 | LLM Answer Generation | Generate grounded answers (hybrid / specific / agentic) |
| 6 | Evaluation | Score answers with an LLM-as-a-Judge |


---


> **Tip:** Run cells top-to-bottom the first time. Each section starts with a short explanation, followed by code, and ends with a quick **test cell** so you can confirm everything works before moving on.

---
## Before You Start — Create a Virtual Environment

A **virtual environment** is an isolated Python installation just for this project. It keeps the packages you install here from clashing with anything else on your computer.

**Run these commands once in your terminal, before opening this notebook:**

#### Linux / macOS

```bash
cd wawopensearch3_hackathon/module1_web_and_llm

python3 -m venv opensearch_hackathon

source opensearch_hackathon/bin/activate

pip install -r requirements.txt

jupyter notebook module1_notebook.ipynb
```

#### Windows (Command Prompt)

```cmd
cd wawopensearch3_hackathon\module1_web_and_llm

python -m venv opensearch_hackathon

opensearch_hackathon\Scripts\activate.bat

pip install -r requirements.txt

jupyter notebook module1_notebook.ipynb
```

#### Windows (PowerShell)

```powershell
cd wawopensearch3_hackathon\module1_web_and_llm

python -m venv opensearch_hackathon

opensearch_hackathon\Scripts\Activate.ps1

pip install -r requirements.txt

jupyter notebook module1_notebook.ipynb
```

> If PowerShell blocks the activation script, run this first:  
> `Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser`

Once the notebook is open, run the cell below to confirm you are inside the right environment and all packages are installed, or simply use jupyter notebook ad select the corresponding kernel from the top right.

In [1]:
# ── Environment check — run this first
import sys, subprocess, pathlib, importlib

# 1. Show which Python / venv is active
print(f"Python executable: {sys.executable}")
print(f"Python version:    {sys.version.split()[0]}")

in_correct_venv = "opensearch_hackathon" in sys.executable
if in_correct_venv:
    print("Virtual env:       ✅ opensearch_hackathon is active")
else:
    print("Virtual env:       ⚠️  opensearch_hackathon not detected")
    print("   Make sure you ran:  source opensearch_hackathon/bin/activate")
    print("   (Windows:           opensearch_hackathon\\Scripts\\activate.bat)")
    print("   Then relaunch Jupyter from that terminal.")

# 2. Install dependencies
req_file = pathlib.Path("requirements.txt")
assert req_file.exists(), "❌ requirements.txt not found — are you in the right folder?"

print("\nInstalling requirements...")
for cmd in [
    [sys.executable, "-m", "pip", "install", "--upgrade", "setuptools"],
    [sys.executable, "-m", "pip", "install", "-r", str(req_file), "--quiet"],
    [sys.executable, "-m", "pip", "install", "-e", ".", "--quiet"],
]:
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("pip STDERR:\n", r.stderr)
        raise RuntimeError(f"❌ Failed: {' '.join(cmd)}")
print("✅ All packages installed")

# 3. Quick import check
required = ["langchain", "langchain_mistralai", "mistralai",
            "dotenv", "bs4", "tqdm", "pandas", "requests"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    print(f"\n⚠️  Still missing: {missing}")
    print("   Restart the kernel (Kernel → Restart) and re-run this cell.")
else:
    print("✅ All key packages importable — you\'re good to go!")

Python executable: /Users/roxanneelbaff/Documents/projects/gitlab/wawopensearch3_hackathon/module1_web_and_llm/opensearch_hackathon/bin/python
Python version:    3.11.9
Virtual env:       ✅ opensearch_hackathon is active

Installing requirements...
✅ All packages installed
✅ All key packages importable — you're good to go!


---
## Section 0 — Setup & Sanity Check

Before doing anything else we need to:
1. Confirm you are running Python 3.10 or newer.
2. Load your Mistral API key from a `.env` file.
3. Send a test message with **both model sizes** to confirm they respond.

**What is a `.env` file?**  
It is a plain text file that stores secrets (like API keys) *outside* your code so they are never accidentally shared. Create a file named `.env` in the **same folder** as this notebook and add the following lines:

```
LLM_API_KEY=<your-key-here>
```

The `.env` file is already listed in `.gitignore`, so it will never be committed to the repository.

**Which model size should I use?**

| | `small` | `large` |
|---|---|---|
| Speed | Faster | Slower |
| Cost | Lower | Higher |
| Best for | Retrieval, summarisation, evaluation | Complex reasoning, nuanced answers |



In [2]:
# 0.1 Check Python version
import sys

print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 10), "Python 3.10 or newer is required. Please upgrade."
print("✅ Python version OK")

Python version: 3.11.9 (v3.11.9:de54cf5be3, Apr  2 2024, 07:12:50) [Clang 13.0.0 (clang-1300.0.29.30)]
✅ Python version OK


In [3]:
# 0.2  Load environment variables from .env 
import os
from dotenv import load_dotenv

load_dotenv()  # reads the .env file in the current directory

LLM_API_KEY    = os.getenv("LLM_API_KEY")
LLM_MODEL = "mistral-small-2603"
LLM_MODEL_LARGE = "mistral-large-2512"

assert LLM_API_KEY, "❌ LLM_API_KEY not found — create a .env file (see instructions above)"
print(f"✅ API key loaded (ends with ...{LLM_API_KEY[-4:]})")
print(f"   Small model: {LLM_MODEL}")
print(f"   Large model: {LLM_MODEL_LARGE}")

✅ API key loaded (ends with ...79ym)
   Small model: mistral-small-2603
   Large model: mistral-large-2512


In [4]:
# TEST — confirm both model sizes respond 
from langchain.messages import HumanMessage
from langchain_mistralai import ChatMistralAI

llm_small = ChatMistralAI(
    api_key=LLM_API_KEY,
    model=LLM_MODEL,
    temperature=0.7,
)

llm_large = ChatMistralAI(
    api_key=LLM_API_KEY,
    model=LLM_MODEL_LARGE,
    temperature=0.7,
)


def test_llm_connection() -> None:
    """Send a hello-world message with each model size and print the replies."""
    msg = [HumanMessage(content="Hi! We are in a cool hackathon")]
    for name, model in [("small", llm_small),]: #("large", llm_large)
        reply = model.invoke(msg).content
        print(f"[{name}] {reply}")

test_llm_connection()
print("\n✅ Both model sizes work — ready to proceed!")

[small] That sounds awesome! Hackathons are such a great way to collaborate, innovate, and push creative boundaries. What’s the theme or focus of your hackathon? Are you building something tech-related, social impact-driven, or just experimenting with wild ideas? 🚀

(Also, if you need brainstorming help, debugging, or just a pep talk, I’m here for it!) 😊

✅ Both model sizes work — ready to proceed!
